# 02: Modern TensorFlow / Keras 3 Step-by-Step Implementation Guide
### (Latest Standards: tf.data Pipelines, Functional API, Callbacks & Evaluation)

මෙම Notebook එකෙහි අරමුණ වන්නේ Notebook 01 හි සාකච්ඡා කළ Datasets 3 සඳහා තෝරාගත් Deep Learning Techniques Modern TensorFlow 2.x / Keras 3 හි නවතම standard ප්‍රමිතීන්ට අනුකූලව Step-by-step ක්‍රියාත්මක කරන ආකාරය පෙන්වා දීමයි.

අපි මෙහිදී පහත සඳහන් Industry Standard Best Practices භාවිත කරමු:
1. tf.data.Dataset High-Performance Pipelines (cache, prefetch, AUTOTUNE)
2. Keras In-Model Preprocessing Layers (Rescaling, TextVectorization, Normalization)
3. Keras Functional API Architecture (Layer-by-Layer Detailed Rationale)
4. Production Callbacks (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)
5. Model Evaluation & Visualizations


--- 
## 1. Computer Vision: CNN Model for Fashion-MNIST

### Step 1.1: Imports & TensorFlow Configuration


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", bool(tf.config.list_physical_devices('GPU')))


TensorFlow Version: 2.21.0
GPU Available: False


### Step 1.2: tf.data High-Performance Pipeline එක නිර්මාණය කිරීම

TensorFlow හි දත්ත ඉතා ඉක්මනින් (Optimized) GPU එකට ලබාදීම සඳහා tf.data.Dataset යොදාගනී. 
- cache(): පළමු epoch එකෙන් පසු data memory එකෙහි තබා ගනී.
- prefetch(tf.data.AUTOTUNE): Model එක train වන අතරතුර ඊළඟ batch එක සූදානම් කරයි (CPU-GPU overlap).


In [2]:
# Load Data
(X_train_img, y_train_img), (X_test_img, y_test_img) = tf.keras.datasets.fashion_mnist.load_data()

# Reshape data to add channel dimension (28, 28) -> (28, 28, 1)
X_train_img = np.expand_dims(X_train_img, axis=-1)
X_test_img = np.expand_dims(X_test_img, axis=-1)

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# Train Pipeline
train_ds_img = tf.data.Dataset.from_tensor_slices((X_train_img, y_train_img))
train_ds_img = train_ds_img.shuffle(10000).batch(BATCH_SIZE).cache().prefetch(buffer_size=AUTOTUNE)

# Test Pipeline
test_ds_img = tf.data.Dataset.from_tensor_slices((X_test_img, y_test_img))
test_ds_img = test_ds_img.batch(BATCH_SIZE).cache().prefetch(buffer_size=AUTOTUNE)

print("tf.data Pipelines created successfully!")


tf.data Pipelines created successfully!


### Step 1.3: CNN Architecture ගොඩනැගීම (Keras Functional API)


In [3]:
def build_cnn_model():
    inputs = layers.Input(shape=(28, 28, 1), name="image_input")
    
    # 1. In-Model Preprocessing & Data Augmentation
    x = layers.Rescaling(1.0 / 255.0, name="scaling")(inputs)
    x = layers.RandomFlip("horizontal", name="augmentation")(x)
    
    # 2. Convolutional Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)
    
    # 3. Convolutional Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)
    
    # 4. Dense Classifier
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(128, activation="relu", name="dense1")(x)
    x = layers.Dropout(0.3, name="dropout1")(x)
    outputs = layers.Dense(10, activation="softmax", name="predictions")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Fashion_CNN")
    return model

cnn_model = build_cnn_model()
cnn_model.summary()


Model: "Fashion_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scaling (Rescaling)             │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (RandomFlip)       │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 28, 28, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 14, 14, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 422,026 (1.61 MB)

 Trainable params: 421,834 (1.61 MB)

 Non-trainable params: 192 (768.00 B)

### CNN Architecture: Layer-by-Layer Detailed Rationale (ඇයි මේ එක එක Layer එක යෙදුවේ?)

1. **`Input(shape=(28, 28, 1))`:**
   - **ඇයි යෙදුවේ?** Fashion-MNIST හි සෑම රූපයක්ම පික්සල් 28x28 Grayscale (1 channel) වේ. Model එකට ඇතුළුවන Tensor එකෙහි නියමිත ආකෘතිය පැහැදිලිව නිර්වචනය කිරීමට මෙය යොදාගනී.

2. **`Rescaling(1.0 / 255.0)`:**
   - **ඇයි යෙදුවේ?** Pixel අගයන් 0 සිට 255 දක්වා පැතිර පවතී. ඒවා 0.0 සිට 1.0 දක්වා Scale කිරීමෙන් Backpropagation හිදී Gradients අධික ලෙස විශාල වීම (Exploding) වැළකී Gradient Descent වේගවත් වේ.

3. **`RandomFlip('horizontal')`:**
   - **ඇයි යෙදුවේ?** Data Augmentation සඳහාය. ඇඳුමක් තිරස් අතට මාරු වුවද (Horizontal flip) එහි වර්ගය (Class) වෙනස් නොවේ. එමගින් Model එකට නව රූප වෙනස්කම් ඉගෙන ගැනීමට සැලැස්වීමෙන් Overfitting පාලනය වේ.

4. **`Conv2D(32, (3, 3), padding='same', activation='relu')`:**
   - **ඇයි 32 Filters?** පළමු layer එකෙන් රූපයේ මූලික ලක්ෂණ (Low-level features: දාර, ඉරි, කෙළවරවල්) 32ක් ඉගෙන ගනී.
   - **ඇයි (3, 3) Kernel Size?** රූපයේ ලඟින් පිහිටි පික්සල් 3x3 ප්‍රදේශයක spatial relationship එක ග්‍රහණය කරගැනීමට කුඩා kernel එකක් ඉතා කාර්යක්ෂම වේ.
   - **ඇයි padding='same'?** Convolution සිදුකිරීමේදී රූපයේ දාරවල පික්සල් අහිමි වී 28x28 ප්‍රමාණය කුඩා වීම වැළැක්වීමටය.
   - **ඇයි activation='relu'?** Non-linearity එකතු කිරීමටය. Vanishing Gradient problem එක වළක්වයි.

5. **`BatchNormalization()`:**
   - **ඇයි යෙදුවේ?** Conv2D layer එකෙන් ලැබෙන Feature maps වල activations Normalize කරයි. එමගින් Model එකෙහි Training වේගවත් වී, ඉහළ Learning rate එකක් භාවිත කිරීමට හැකි වේ.

6. **`MaxPooling2D((2, 2))`:**
   - **ඇයි යෙදුවේ?** Spatial dimensions 28x28 සිට 14x14 දක්වා අඩකින් අඩු කරයි (75% dimensionality reduction). එමගින් Computation ඉතිරි වන අතර, වස්තුවේ පිහිටීම සුළු වශයෙන් වෙනස් වුවද හඳුනාගැනීමේ හැකියාව (Translation Invariance) ලබාදෙයි.

7. **`Conv2D(64, (3, 3), padding='same', activation='relu')`:**
   - **ඇයි 64 Filters?** ගැඹුරු layers වලදී මූලික දාර එකතු වී සාදන සංකීර්ණ රටා (High-level features: සාක්කු, කරවල්, පාවහන් පටි) 64ක් ඉගෙන ගැනීමට filters ගණන දෙගුණ කරයි.

8. **`MaxPooling2D((2, 2))`:**
   - **ඇයි යෙදුවේ?** 14x14 feature maps නැවත 7x7 දක්වා කුඩා කරයි.

9. **`Flatten()`:**
   - **ඇයි යෙදුවේ?** 3D Tensor එකක් වන (7, 7, 64) -> 1D Feature Vector එකක් (7 * 7 * 64 = 3136 features) බවට පත්කර Fully Connected Dense classification layers වලට සූදානම් කරයි.

10. **`Dense(128, activation='relu')`:**
    - **ඇයි යෙදුවේ?** Flatten කරන ලද 3136 features අතර පවතින සංකීර්ණ non-linear සම්බන්ධතා ඉගෙන ගෙන classification සඳහා තීරණ ගනී.

11. **`Dropout(0.3)`:**
    - **ඇයි 0.3 Rate එකක්?** Training අතරතුර අහඹු ලෙස Neurons 30%ක් අක්‍රිය කරයි. එමගින් Model එක එකම Neurons මත අධික ලෙස යැපීම (Co-adaptation) වැළකී Overfitting පාලනය වේ.

12. **`Dense(10, activation='softmax')`:**
    - **ඇයි 10 Units & Softmax?** Fashion-MNIST හි Classes 10ක් ඇත. Softmax activation මගින් සෑම Class එකකටම අයත් වීමේ සම්භාවිතාවය (Probabilities: 0.0 - 1.0) ලබාදෙන අතර, එකතුව 1.0ක් වේ.


### Step 1.4: Model Compilation & Callbacks


In [ ]:
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

img_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

history_cnn = cnn_model.fit(
    train_ds_img,
    validation_data=test_ds_img,
    epochs=10,
    callbacks=img_callbacks
)


In [ ]:
# Step 1.5: Results Evaluation & Visualizations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_cnn.history['accuracy'], label='Train Accuracy')
axes[0].plot(history_cnn.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('CNN Accuracy Curves')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_cnn.history['loss'], label='Train Loss')
axes[1].plot(history_cnn.history['val_loss'], label='Val Loss')
axes[1].set_title('CNN Loss Curves')
axes[1].legend()
axes[1].grid(True)

plt.show()


--- 
## 2. Natural Language Processing: Bi-LSTM Model for SMS Spam Detection

### Step 2.1: Data Load & TextVectorization Adaptation


In [2]:
# Load SMS Data
url_sms = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df_sms = pd.read_csv(url_sms, sep='\t', header=None, names=['label', 'message'])
df_sms['target'] = df_sms['label'].map({'ham': 0, 'spam': 1})

from sklearn.model_selection import train_test_split
# Robust indexing for Pandas extension / PyArrow arrays via .astype(str).tolist()
X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    df_sms['message'].astype(str).tolist(), 
    df_sms['target'].to_numpy(), 
    test_size=0.2, 
    random_state=42, 
    stratify=df_sms['target'].to_numpy()
)

MAX_VOCAB = 5000
MAX_SEQ_LEN = 50

vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode='int',
    output_sequence_length=MAX_SEQ_LEN
)
vectorizer.adapt(X_train_txt)
print("TextVectorization Adapted Successfully!")


TextVectorization Adapted Successfully!


In [5]:
train_ds_txt = tf.data.Dataset.from_tensor_slices((X_train_txt, y_train_txt)).shuffle(1000).batch(32).prefetch(AUTOTUNE)
test_ds_txt = tf.data.Dataset.from_tensor_slices((X_test_txt, y_test_txt)).batch(32).prefetch(AUTOTUNE)

def build_bilstm_model():
    inputs = layers.Input(shape=(1,), dtype=tf.string, name="text_input")
    x = vectorizer(inputs)
    x = layers.Embedding(input_dim=MAX_VOCAB, output_dim=64, mask_zero=True, name="embedding")(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=False), name="bilstm")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    x = layers.Dense(32, activation="relu", name="dense1")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="predictions")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="SMS_BiLSTM")

nlp_model = build_bilstm_model()
nlp_model.summary()


Model: "SMS_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 50)        │          0 │ text_input[0][0]  │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 64)    │    320,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ (None, 64)        │     24,832 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ bilstm[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, 32)        │      2,080 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ predictions (Dense) │ (None, 1)         │         33 │ dense1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 346,945 (1.32 MB)

 Trainable params: 346,945 (1.32 MB)

 Non-trainable params: 0 (0.00 B)

### Bi-LSTM Architecture: Layer-by-Layer Detailed Rationale (ඇයි මේ එක එක Layer එක යෙදුවේ?)

1. **`Input(shape=(1,), dtype=tf.string)`:**
   - **ඇයි යෙදුවේ?** raw string text (e.g., 'Win 1000$ cash prize now!') සෘජුවම model එකට ඇතුළත් කිරීමට.

2. **`TextVectorization(max_tokens=5000, output_sequence_length=50)`:**
   - **ඇයි යෙදුවේ?** පෙළ දත්ත Integer Token IDs බවට පත් කරයි (lowercase කිරීම, punctuation ඉවත් කිරීම, සහ වචන වලට 1-5000 දක්වා අංක ලබාදීම). `output_sequence_length=50` මගින් වචන 50කට වඩා වැඩි පණිවිඩ කපා දමන අතර (Truncate), අඩු පණිවිඩ වලට zero-padding එකතු කරයි.

3. **`Embedding(input_dim=5000, output_dim=64, mask_zero=True)`:**
   - **ඇයි 64 Vector Dim & mask_zero=True?** සෑම වචන අංකයක් සඳහාම අර්ථවත් 64-dimensional continuous dense vector එකක් සාදයි (e.g., 'win' සහ 'cash' අතර අර්ථමය සහසම්බන්ධතාවය ඉගෙන ගනී). `mask_zero=True` මගින් zero-padding tokens LSTM ගණනය කිරීම්වලින් ඉවත් කර Memory & Computation ඉතිරි කරයි.

4. **`Bidirectional(LSTM(32, return_sequences=False))`:**
   - **ඇයි Bidirectional & 32 Units?** Forward LSTM මගින් වාක්‍යය මුල සිට අගටද, Backward LSTM මගින් අග සිට මුලටද කියවයි (32 units + 32 units = 64 outputs). 'Free entry call now' වැනි වාක්‍යයක වචනවල දෙපසම පවතින සන්දර්භය (Context) තේරුම් ගනී.

5. **`Dropout(0.3)`:**
   - **ඇයි යෙදුවේ?** විශේෂිත වචන එකතු (Specific word combinations) මත Model එක අධික ලෙස යැපීම (Overfitting) වැළැක්වීමට.

6. **`Dense(32, activation='relu')`:**
   - **ඇයි යෙදුවේ?** LSTM එකෙන් ලැබෙන contextual summary vector එක තවත් පිරිසිදු කර classification සඳහා සැකසීමට.

7. **`Dense(1, activation='sigmoid')`:**
   - **ඇයි 1 Unit & Sigmoid?** Binary classification (Spam ද Ham ද) සඳහාය. Output එක 0.0 සිට 1.0 දක්වා සම්භාවිතාවයක් (Probability) ලබාදෙයි (0.5ට වැඩි නම් Spam වේ).


In [ ]:
nlp_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

history_nlp = nlp_model.fit(
    train_ds_txt,
    validation_data=test_ds_txt,
    epochs=6,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]
)


Epoch 1/6


/Users/adithyabandara/miniconda3/envs/stt/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


In [ ]:
# Step 2.4: Inference Test Function (Real-world Usage)
def predict_sms(sample_messages):
    preds = nlp_model.predict(np.array(sample_messages))
    for msg, prob in zip(sample_messages, preds):
        label = "SPAM" if prob[0] > 0.5 else "HAM"
        print(f"[{label} ({prob[0]:.4f})] -> {msg}")

test_samples = [
    "URGENT! You have won a 1000$ cash prize. Call now to claim!",
    "Hey friend, are we still meeting for tea this evening?",
    "Free entry in 2 a wk weekly comp to win FA Cup final tkts 21st May 2005."
]
predict_sms(test_samples)


--- 
## 3. Tabular Data: MLP Regression Model for California Housing

### Step 3.1: Data Load & Preprocessing


In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_tab, X_test_tab, y_train_tab, y_test_tab = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)

norm_layer = layers.Normalization(axis=-1, name="feature_normalization")
norm_layer.adapt(X_train_tab)

train_ds_tab = tf.data.Dataset.from_tensor_slices((X_train_tab, y_train_tab)).shuffle(5000).batch(128).prefetch(AUTOTUNE)
test_ds_tab = tf.data.Dataset.from_tensor_slices((X_test_tab, y_test_tab)).batch(128).prefetch(AUTOTUNE)

def build_mlp_model():
    inputs = layers.Input(shape=(X_train_tab.shape[1],), name="tabular_input")
    x = norm_layer(inputs)
    x = layers.Dense(128, activation="relu", name="dense1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Dropout(0.2, name="dropout1")(x)
    x = layers.Dense(64, activation="relu", name="dense2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Dense(32, activation="relu", name="dense3")(x)
    outputs = layers.Dense(1, name="house_value")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="Housing_MLP")

mlp_model = build_mlp_model()
mlp_model.summary()


Model: "Housing_MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tabular_input (InputLayer)      │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_normalization           │ (None, 8)              │            17 │
│ (Normalization)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (Dense)                  │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense3 (Dense)                  │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ house_value (Dense)             │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,306 (48.07 KB)

 Trainable params: 11,905 (46.50 KB)

 Non-trainable params: 401 (1.57 KB)

### Tabular MLP Architecture: Layer-by-Layer Detailed Rationale (ඇයි මේ එක එක Layer එක යෙදුවේ?)

1. **`Input(shape=(8,))`:**
   - **ඇයි යෙදුවේ?** California Housing dataset හි පවතින numerical features 8 (MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude) ලබාගැනීමට.

2. **`norm_layer` (Keras Normalization):**
   - **ඇයි යෙදුවේ?** Features වල Scales එකිනෙකට ඉතා වෙනස්ය (e.g. MedInc 0.5-15 vs Population 3-35,000). Standardization (`mean=0, std=1`) මගින් සියලුම features එකම පරාසයකට පත්කර Weights එක හා සමාන ලෙස update වීමට සලස්වයි.

3. **`Dense(128, activation='relu')`:**
   - **ඇයි 128 Units?** පළමු layer එකෙහි high capacity එකක් තැබීමෙන් features 8 අතර පවතින සංකීර්ණ non-linear හවුල් රටා ඉගෙන ගනී.

4. **`BatchNormalization()` & `Dropout(0.2)`:**
   - **ඇයි යෙදුවේ?** Internal covariate shift පාලනය කර training වේගවත් කිරීමට BatchNorm ද, Tabular weights මත overfitting වැළැක්වීමට 20% Dropout ද යොදනු ලැබේ.

5. **`Dense(64, activation='relu')` & `Dense(32, activation='relu')`:**
   - **ඇයි යෙදුවේ?** 128 සිට 64 සහ 32 දක්වා layers ප්‍රමාණය කුඩා කිරීමෙන් (Funnel architecture) ඉගෙන ගත් සංකීර්ණ ලක්ෂණ වඩාත් පිරිසිදු කෙටුම්පතක් (High-level representation) බවට පත්කරයි.

6. **`Dense(1)` (Linear Output):**
   - **ඇයි 1 Unit & No Activation?** Continuous House Price ($100,000 units වලින්) පුරෝකථනය කරන Regression Task එකක් බැවින් Activation Function එකක් නොමැතිව (Linear Output) එක 1 Output Unit එකක් යොදනු ලැබේ.


In [ ]:
mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), 'mae']
)

history_mlp = mlp_model.fit(
    train_ds_tab,
    validation_data=test_ds_tab,
    epochs=15,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
)


Epoch 1/15


/Users/adithyabandara/miniconda3/envs/stt/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


In [ ]:
# Step 3.4: Evaluation & Prediction Comparison
eval_results = mlp_model.evaluate(test_ds_tab)
print(f"\nTest MSE Loss: {eval_results[0]:.4f}")
print(f"Test RMSE: {eval_results[1]:.4f}")
print(f"Test MAE: {eval_results[2]:.4f}")

# Actual vs Predicted Plot
y_pred_tab = mlp_model.predict(test_ds_tab).flatten()

plt.figure(figsize=(7, 6))
plt.scatter(y_test_tab, y_pred_tab, alpha=0.3, color='crimson')
plt.plot([0, 5], [0, 5], 'k--', lw=2)
plt.xlabel("Actual House Price ($100k)")
plt.ylabel("Predicted House Price ($100k)")
plt.title("Tabular MLP: Actual vs Predicted")
plt.grid(True)
plt.show()
